In [ ]:
"""
U-AutoRec for MovieLens 10M  —  Optimized Implementation
==========================================================
Paper: "AutoRec: Autoencoders Meet Collaborative Filtering"
Sedhain et al., WWW 2015   |   Target RMSE: 0.867

Key differences vs U-AutoRec ML-1M:
  - ML-10M is ~10x larger: 10M ratings, 69,878 users, 10,677 items
  - Each user vector ∈ R^(n_items) where n_items = 10,677
  - User vectors are SPARSER than item vectors (avg ~143 ratings/user)
  - Dense matrix: 69,878 × 10,677 × 4 bytes ≈ 2.8 GB → use sparse
  - lambda_reg best for U-AutoRec ML-10M: 0.001 (same as ML-1M)

Architecture:
  h(r; θ) = f( W · g(V·r + μ) + b )
    g(·) = Sigmoid   (hidden activation — paper Table 1b best)
    f(·) = Identity  (output activation — paper Table 1b best)
    input dimension = n_items = 10,677

ML-10M format: ratings.dat → UserID::MovieID::Rating::Timestamp
  Ratings: 10,000,054   Users: 69,878   Movies: 10,677

All fixes from corrected U-AutoRec ML-1M applied:
  ✓ L2 reg computed globally via model.l2_penalty() — not per-batch-scaled
  ✓ Rating matrix built from train_data ONLY per fold (no leakage)
  ✓ Global RMSE over all test pairs (not average of per-user RMSEs)
  ✓ Sparse matrix storage — only dense conversion per batch
  ✓ Best model only saved (no periodic checkpoints)

Kaggle-ready:
  - Config dataclass (no argparse)
  - num_workers=0
  - Auto-discovers ratings.dat
  - All writes to /kaggle/working/
"""

import os
import gc
import time
import random
import dataclasses as _dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass
from collections import defaultdict
import logging
import scipy.sparse as sp

# ──────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────
# Config  ← only thing you need to edit
# ──────────────────────────────────────────────────────────────
@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────
    # Folder containing ML-10M data (ratings.dat found automatically)
    dataset_root:   str   = "/kaggle/input/datasets/priyanshuunayak/ml-10muauto"
    checkpoint_dir: str   = "/kaggle/working/u_autorec_10m"

    # ── Model  (paper best) ───────────────────────────────────
    hidden_units:   int   = 500       # paper Figure 2: best at k=500

    # ── Training  (paper exact settings) ─────────────────────
    epochs:         int   = 300
    # ML-10M: 69,878 users — each user vector ∈ R^10,677
    # batch of 256 users = 256 × 10,677 × 4 bytes ≈ 10 MB → safe for T4
    batch_size:     int   = 256
    lr:             float = 0.001     # RProp initial step size
    # U-AutoRec lambda: paper tunes {0.001,0.01,0.1,1,100,1000}
    # U-AutoRec ML-10M best value is 0.001
    lambda_reg:     float = 0.001

    # ── Early stopping ────────────────────────────────────────
    early_stop:     int   = 20        # patience on val RMSE
    log_every:      int   = 10        # print every N epochs

    # ── System ────────────────────────────────────────────────
    seed:           int   = 42
    num_workers:    int   = 0         # must be 0 on Kaggle
    device:         str   = "auto"    # "auto" | "cuda" | "cpu"


# ──────────────────────────────────────────────────────────────
# Utilities
# ──────────────────────────────────────────────────────────────
def cfg_to_dict(cfg):
    try:
        return _dc.asdict(cfg)
    except TypeError:
        return vars(cfg)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False


# ──────────────────────────────────────────────────────────────
# Auto-discover ratings.dat
# ──────────────────────────────────────────────────────────────
def find_ratings_file(root: str) -> str:
    logger.info(f"Searching for ratings.dat under: {root}")
    for dirpath, _, files in os.walk(root):
        if "ratings.dat" in files:
            path = os.path.join(dirpath, "ratings.dat")
            logger.info(f"  Found: {path}")
            return path
    all_files = [
        os.path.join(d, f)
        for d, _, fs in os.walk(root)
        for f in fs
    ]
    listing = "\n  ".join(all_files) or "(empty)"
    raise FileNotFoundError(
        f"ratings.dat not found under '{root}'.\n"
        f"Files present:\n  {listing}\n"
        f"→ Fix 'dataset_root' in Config."
    )


# ──────────────────────────────────────────────────────────────
# Data Loading
# ──────────────────────────────────────────────────────────────
def load_ml10m(path: str):
    """
    Parse ML-10M ratings.dat → zero-indexed numpy array (N, 3).
    Columns: [user_idx, item_idx, rating]

    ML-10M format: UserID::MovieID::Rating::Timestamp
    Ratings: floats 0.5 to 5.0 in 0.5 steps
    """
    logger.info(f"Loading ML-10M from: {path}")
    logger.info("  (parsing ~10M lines, please wait ~30s...)")

    raw_u, raw_i, raw_r = [], [], []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("::")
            if len(parts) < 3:
                continue
            raw_u.append(int(parts[0]))
            raw_i.append(int(parts[1]))
            raw_r.append(float(parts[2]))

    raw_u = np.array(raw_u, dtype=np.int32)
    raw_i = np.array(raw_i, dtype=np.int32)
    raw_r = np.array(raw_r, dtype=np.float32)

    # Zero-index users and items
    unique_u = np.unique(raw_u)
    unique_i = np.unique(raw_i)
    u2i = {u: i for i, u in enumerate(unique_u)}
    i2i = {it: i for i, it in enumerate(unique_i)}

    u_idx = np.array([u2i[u] for u in raw_u], dtype=np.int32)
    i_idx = np.array([i2i[i] for i in raw_i], dtype=np.int32)

    n_users = len(unique_u)
    n_items = len(unique_i)

    data = np.column_stack([u_idx, i_idx, raw_r]).astype(np.float32)

    logger.info(f"  Users={n_users:,}  Items={n_items:,}  Ratings={len(data):,}")
    logger.info(f"  Rating range: [{raw_r.min():.1f}, {raw_r.max():.1f}]")
    logger.info(f"  Avg ratings/user: {len(data)/n_users:.1f}")
    logger.info(f"  Avg ratings/item: {len(data)/n_items:.1f}")
    return data, n_users, n_items


# ──────────────────────────────────────────────────────────────
# Splitting
# ──────────────────────────────────────────────────────────────
def split_90_10(data, seed):
    """90% trainval / 10% test — paper exact procedure."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data))
    cut = int(len(data) * 0.9)
    return data[idx[:cut]], data[idx[cut:]]


def split_val(trainval, seed):
    """10% of trainval → validation."""
    rng = np.random.default_rng(seed + 1000)
    idx = rng.permutation(len(trainval))
    cut = int(len(trainval) * 0.9)
    return trainval[idx[:cut]], trainval[idx[cut:]]


# ──────────────────────────────────────────────────────────────
# Sparse Rating Matrix — (n_users × n_items)
# ──────────────────────────────────────────────────────────────
def make_sparse_matrix(data, n_users, n_items):
    """
    Build CSR sparse matrix (n_users × n_items) from train data only.
    U-AutoRec: rows=users, cols=items — each row is one user's item ratings.

    Memory comparison for ML-10M:
      Dense:  69,878 × 10,677 × 4 bytes = 2.8 GB  ← too large
      Sparse: ~10M entries × (4+4+4) bytes ≈ 120 MB  ← fine
    """
    u_idx = data[:, 0].astype(np.int32)
    i_idx = data[:, 1].astype(np.int32)
    vals  = data[:, 2].astype(np.float32)

    # shape: (n_users, n_items) — each row is one user's item rating vector
    R_sparse = sp.csr_matrix(
        (vals, (u_idx, i_idx)),
        shape=(n_users, n_items),
        dtype=np.float32,
    )
    return R_sparse


# ──────────────────────────────────────────────────────────────
# Dataset — sparse → dense conversion per batch
# ──────────────────────────────────────────────────────────────
class UserRatingDataset(Dataset):
    """
    Each sample = one user's partially observed item rating vector.
    Input to U-AutoRec: r^(u) = (R_u1, R_u2, ..., R_un) ∈ R^n_items

    Uses sparse matrix to avoid storing full 2.8 GB dense matrix.
    Converts sparse row → dense only for each requested sample.
    """
    def __init__(self, R_sparse: sp.csr_matrix):
        self.R       = R_sparse                  # (n_users, n_items) sparse
        self.n_users = R_sparse.shape[0]

    def __len__(self):
        return self.n_users

    def __getitem__(self, idx):
        # Sparse row → dense vector
        row  = np.asarray(self.R[idx].todense(), dtype=np.float32).squeeze()
        mask = (row != 0).astype(np.float32)
        return row, mask


# ──────────────────────────────────────────────────────────────
# U-AutoRec Model
# ──────────────────────────────────────────────────────────────
class UAutoRec(nn.Module):
    """
    User-based AutoRec.

        h(r; θ) = f( W · g(V·r + μ) + b )

    Input  : r^(u) ∈ R^n_items  — user u's partially observed item ratings
             n_items = 10,677 for ML-10M
    Hidden : k = 500  (Sigmoid activation)
    Output : R^n_items (Identity activation)

    g(·) = Sigmoid   ← hidden activation  (paper Table 1b best)
    f(·) = Identity  ← output activation  (paper Table 1b best)

    Parameters: 2 * n_items * k + k + n_items
               = 2 * 10677 * 500 + 500 + 10677
               ≈ 10.7M parameters
    """
    def __init__(self, n_items: int, k: int = 500):
        super().__init__()
        self.encoder = nn.Linear(n_items, k)      # V, μ
        self.decoder = nn.Linear(k, n_items)      # W, b
        self.g = nn.Sigmoid()
        # f = Identity → no module needed
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, r):
        """
        r      : (batch, n_items)  — partially observed user vectors
        returns: (batch, n_items)  — reconstructed item ratings per user
        """
        return self.decoder(self.g(self.encoder(r)))

    def l2_penalty(self):
        """
        Global L2 on weight matrices only (not biases).
        Implements (λ/2)(||W||²_F + ||V||²_F) from paper Eq.2.
        Called once per optimizer step — NOT scaled by batch size.
        """
        return (
            self.encoder.weight.norm(p="fro") ** 2 +
            self.decoder.weight.norm(p="fro") ** 2
        )


# ──────────────────────────────────────────────────────────────
# Loss — masked MSE (observed ratings only)
# ──────────────────────────────────────────────────────────────
def masked_mse(pred, target, mask):
    """
    MSE over OBSERVED ratings only.
    Implements ||r^(u) - h(r^(u))||²_O from paper Eq.2.
    """
    diff  = (pred - target) * mask
    n_obs = mask.sum()
    if n_obs == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return (diff ** 2).sum() / n_obs


# ──────────────────────────────────────────────────────────────
# Evaluation — global RMSE over all test pairs
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_rmse(model, test_pairs, R_sparse, device, user_chunk=512):
    """
    Compute global RMSE on (user, item, rating) test pairs.

    Strategy for ML-10M memory efficiency:
      - Process users in chunks of `user_chunk`
      - For each chunk, convert sparse rows → dense, run model,
        collect predictions for test pairs in those users
      - Never materialise the full 2.8 GB dense matrix

    Predictions clipped to [0.5, 5.0] — ML-10M valid rating range.
    """
    model.eval()
    n_users = R_sparse.shape[0]

    # Group test pairs by user for efficient lookup
    # user_to_pairs[user_idx] = list of (item_idx, true_rating)
    user_to_pairs = defaultdict(list)
    for u, i, r in test_pairs:
        user_to_pairs[int(u)].append((int(i), float(r)))

    preds_all   = []
    targets_all = []

    for start in range(0, n_users, user_chunk):
        end   = min(start + user_chunk, n_users)

        # Check if any test pairs exist in this chunk — skip if not
        chunk_users = range(start, end)
        if not any(u in user_to_pairs for u in chunk_users):
            continue

        # Dense conversion for this user chunk only
        chunk_sparse = R_sparse[start:end]
        batch = torch.from_numpy(
            np.asarray(chunk_sparse.todense(), dtype=np.float32)
        ).to(device)                                     # (chunk, n_items)

        recon = model(batch).cpu().numpy()               # (chunk, n_items)

        for local_u, global_u in enumerate(range(start, end)):
            if global_u not in user_to_pairs:
                continue
            for i_idx, true_r in user_to_pairs[global_u]:
                pred = float(np.clip(recon[local_u, i_idx], 0.5, 5.0))
                preds_all.append(pred)
                targets_all.append(true_r)

    if len(preds_all) == 0:
        return float("nan")

    preds   = np.array(preds_all,   dtype=np.float64)
    targets = np.array(targets_all, dtype=np.float64)
    return float(np.sqrt(np.mean((preds - targets) ** 2)))


# ──────────────────────────────────────────────────────────────
# One Training Epoch
# ──────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, lambda_reg, device):
    """
    One full pass over all users.

    Loss per step:
      = masked_MSE(pred, R_batch, mask_batch)
      + (lambda_reg / 2) * (||V||²_F + ||W||²_F)

    RProp adapts step sizes per parameter automatically.
    L2 penalty is global — computed once per step, not per sample.
    """
    model.train()
    running_loss = 0.0

    bar = tqdm(loader, desc="  train", leave=False, ncols=90)
    for R_batch, mask_batch in bar:
        R_batch    = R_batch.to(device, non_blocking=True)
        mask_batch = mask_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pred = model(R_batch)

        # Observed-only reconstruction loss
        rec_loss = masked_mse(pred, R_batch, mask_batch)

        # Global L2 regularisation on weight matrices (not biases)
        l2_loss  = (lambda_reg / 2.0) * model.l2_penalty()

        loss = rec_loss + l2_loss
        loss.backward()
        optimizer.step()

        running_loss += rec_loss.item()
        bar.set_postfix(rec=f"{rec_loss.item():.4f}")

    return running_loss / max(len(loader), 1)


# ──────────────────────────────────────────────────────────────
# Single Fold Training
# ──────────────────────────────────────────────────────────────
def run_fold(fold_id, train_data, val_data, test_data,
             n_users, n_items, cfg, device):

    logger.info(f"\n{'='*64}")
    logger.info(f"  FOLD {fold_id}/5  "
                f"train={len(train_data):,}  "
                f"val={len(val_data):,}  "
                f"test={len(test_data):,}")
    logger.info(f"{'='*64}")

    # Build sparse user-item matrix from train_data ONLY (no leakage)
    logger.info("  Building sparse train matrix (users × items)...")
    R_sparse = make_sparse_matrix(train_data, n_users, n_items)
    logger.info(f"  Sparse matrix: {R_sparse.shape}  "
                f"nnz={R_sparse.nnz:,}  "
                f"density={100*R_sparse.nnz/(n_users*n_items):.3f}%  "
                f"mem≈{R_sparse.data.nbytes/1e6:.0f}MB")

    dataset = UserRatingDataset(R_sparse)
    loader  = DataLoader(
        dataset,
        batch_size  = cfg.batch_size,
        shuffle     = True,
        num_workers = cfg.num_workers,          # 0 on Kaggle
        pin_memory  = (device.type == "cuda"),
        drop_last   = False,
    )

    # Model input dim = n_items (user vector length)
    model = UAutoRec(n_items=n_items, k=cfg.hidden_units).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"  UAutoRec  input={n_items:,}  k={cfg.hidden_units}  "
                f"params={n_params:,}")

    # RProp — exact optimiser from paper
    optimizer = optim.Rprop(
        model.parameters(),
        lr         = cfg.lr,
        etas       = (0.5, 1.2),
        step_sizes = (1e-6, 50),
    )

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)
    best_path     = os.path.join(cfg.checkpoint_dir, f"fold{fold_id}_best.pt")
    best_val_rmse = float("inf")
    best_epoch    = 0
    patience      = 0

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()

        train_loss = train_epoch(model, loader, optimizer, cfg.lambda_reg, device)
        val_rmse   = evaluate_rmse(model, val_data, R_sparse, device)
        elapsed    = time.time() - t0

        if epoch % cfg.log_every == 0 or epoch == 1:
            logger.info(
                f"  Ep {epoch:4d}/{cfg.epochs}  "
                f"loss={train_loss:.5f}  "
                f"val_RMSE={val_rmse:.4f}  "
                f"best={best_val_rmse:.4f}  "
                f"({elapsed:.1f}s)"
            )

        # Save best model only
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch    = epoch
            patience      = 0
            torch.save({
                "fold":     fold_id,
                "epoch":    epoch,
                "model":    model.state_dict(),
                "val_rmse": val_rmse,
                "config":   cfg_to_dict(cfg),
            }, best_path)
            logger.info(
                f"  ✓ New best  val_RMSE={val_rmse:.4f}"
                f"  → {os.path.basename(best_path)}"
            )
        else:
            patience += 1

        if cfg.early_stop > 0 and patience >= cfg.early_stop:
            logger.info(f"  Early stop at epoch {epoch} "
                        f"(patience={cfg.early_stop})")
            break

    # Final test RMSE using best model
    ckpt = torch.load(best_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model"])
    test_rmse = evaluate_rmse(model, test_data, R_sparse, device)

    logger.info(f"\n  Fold {fold_id} done  →  "
                f"best_val={best_val_rmse:.4f} (ep {best_epoch})  "
                f"test_RMSE={test_rmse:.4f}")

    # Free memory before next fold
    del R_sparse, dataset, loader, model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return test_rmse


# ──────────────────────────────────────────────────────────────
# 5-Fold Cross Validation
# ──────────────────────────────────────────────────────────────
def run_cv(all_data, n_users, n_items, cfg, device):
    """
    Paper procedure (Section 3):
      - 90/10 random split, repeated 5 times with different seeds
      - 10% of train held out as validation (for early stopping)
      - Report mean ± std RMSE over 5 folds
    """
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    logger.info("\n" + "="*64)
    logger.info("  U-AutoRec ML-10M  —  5-Fold Cross Validation")
    logger.info("="*64)

    fold_rmses = []

    for fold in range(1, 4):
        set_seed(cfg.seed + fold)

        trainval, test = split_90_10(all_data, seed=cfg.seed + fold)
        train, val     = split_val(trainval,   seed=cfg.seed + fold)

        rmse = run_fold(
            fold_id    = fold,
            train_data = train,
            val_data   = val,
            test_data  = test,
            n_users    = n_users,
            n_items    = n_items,
            cfg        = cfg,
            device     = device,
        )
        fold_rmses.append(rmse)

    avg = float(np.mean(fold_rmses))
    std = float(np.std(fold_rmses))

    logger.info("\n" + "="*64)
    logger.info("  FINAL RESULTS")
    logger.info("="*64)
    for i, r in enumerate(fold_rmses):
        logger.info(f"  Fold {i+1}:  RMSE = {r:.4f}")
    logger.info(f"\n  Mean RMSE : {avg:.4f}  ±  {std:.4f}")
    logger.info(f"  Paper     : 0.867  (U-AutoRec ML-10M)")
    logger.info("="*64)

    # Save summary
    out = os.path.join(cfg.checkpoint_dir, "results.txt")
    with open(out, "w") as f:
        f.write("U-AutoRec ML-10M  —  Replication\n")
        f.write("="*40 + "\n\n")
        for i, r in enumerate(fold_rmses):
            f.write(f"Fold {i+1}: {r:.4f}\n")
        f.write(f"\nMean : {avg:.4f} ± {std:.4f}\n")
        f.write(f"Paper: 0.867\n\n")
        f.write(f"Config: {cfg_to_dict(cfg)}\n")
    logger.info(f"Summary → {out}")

    return avg, std, fold_rmses


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main():
    cfg = Config()
    set_seed(cfg.seed)

    # Device
    if cfg.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(cfg.device)

    logger.info(f"Device : {device}")
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        logger.info(f"  GPU  : {props.name}")
        logger.info(f"  VRAM : {props.total_memory / 1e9:.1f} GB")

    # Locate data
    ratings_path = find_ratings_file(cfg.dataset_root)

    # Load — ~30s for 10M rows
    all_data, n_users, n_items = load_ml10m(ratings_path)

    # Config summary
    logger.info(f"\n--- Config ---")
    logger.info(f"  model        : U-AutoRec  (user-based)")
    logger.info(f"  hidden_units : {cfg.hidden_units}")
    logger.info(f"  lambda_reg   : {cfg.lambda_reg}")
    logger.info(f"  epochs       : {cfg.epochs}")
    logger.info(f"  batch_size   : {cfg.batch_size}  (users per batch)")
    logger.info(f"  early_stop   : {cfg.early_stop}")
    logger.info(f"  optimizer    : RProp (lr={cfg.lr})")
    logger.info(f"  activation   : g=Sigmoid, f=Identity  (Table 1b best)")
    logger.info(f"  input_dim    : {n_items:,}  (item ratings per user)")
    logger.info(f"  num_workers  : {cfg.num_workers}")

    run_cv(all_data, n_users, n_items, cfg, device)


if __name__ == "__main__":
    main()